# Day 1 — Gemma 4: execution harness + baseline

**Research:** *Learning to Recover: Teaching Small Coding Agents to Fix Their Own Mistakes Using Execution Feedback*

**Targets:** (1) validated MBPP splits + MBPP+ ID exclusion, (2) runnable test harness, (3) 1–3 attempt agent loop, (4) baseline JSON result files.

**Hardware:** Kaggle GPU T4 ×2. The default 4-bit model is placed on GPU **0** only; 2 T4s are not a single shared memory pool.

**Safety:** This notebook runs generated code in a subprocess with CPU/memory limits, **not** a full sandbox. Run in a disposable environment without secrets; do not share access tokens with the code-runner environment. For robust isolation use an external sandbox/container with network and filesystem restrictions.

**Evaluation note:** Today's percentages are **visible MBPP-test success**, not MBPP+ hidden-test accuracy. Do not train on any MBPP+ task ID.

## A. Environment and packages
Enable GPU and Internet in Kaggle Notebook settings for initial downloads. Avoid updating `torch` unless required; Kaggle already provides a compatible CUDA installation.

In [ ]:
import os, sys, torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
for idx in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(idx)
    print(f"GPU {idx}: {p.name}, {p.total_memory / 1024**3:.2f} GiB")
assert torch.cuda.is_available(), "Select GPU as the Kaggle Accelerator before continuing"

In [ ]:
%pip install -q -U transformers accelerate bitsandbytes datasets

## B. Prepare scripts
These cells write reusable `.py` modules into `/kaggle/working`; they are embedded so this notebook works without separately uploading the scripts.

In [ ]:
%%writefile prepare_datasets.py
"""MBPP preparation for Gemma recovery research.

Official task-id splits: train 601..974; validation 511..600; test 11..510.
Keep test split untouched until frozen evaluation settings are chosen.
"""
from __future__ import annotations

import json
from pathlib import Path
from datasets import load_dataset


def load_mbpp():
    ds = load_dataset("google-research-datasets/mbpp", "full")
    splits = {
        "train": [dict(x) for x in ds["train"]],
        "validation": [dict(x) for x in ds["validation"]],
        "test": [dict(x) for x in ds["test"]],
    }
    expected = {"train": (601, 974, 374), "validation": (511, 600, 90), "test": (11, 510, 500)}
    ids_seen = set()
    for name, (lo, hi, count) in expected.items():
        rows = splits[name]
        ids = {int(x["task_id"]) for x in rows}
        assert len(rows) == count == len(ids), f"Unexpected {name} size: {len(rows)}"
        assert all(lo <= i <= hi for i in ids), f"Unexpected {name} task IDs"
        assert ids.isdisjoint(ids_seen), "Task ID overlap detected!"
        ids_seen.update(ids)
    return splits


def exclude_evalplus_overlap(splits):
    """Prevent training/validation exposure of any EvalPlus MBPP+ task.

    MBPP+ includes some task IDs in original MBPP training/validation ranges.
    Load its *IDs only*; never use its reference solutions/hidden tests in prompts.
    """
    evalplus = load_dataset("evalplus/mbppplus", split="test")
    evalplus_ids = {int(x["task_id"]) for x in evalplus}
    clean = {
        "train": [r for r in splits["train"] if int(r["task_id"]) not in evalplus_ids],
        "validation": [r for r in splits["validation"] if int(r["task_id"]) not in evalplus_ids],
    }
    for name, rows in clean.items():
        assert {int(x["task_id"]) for x in rows}.isdisjoint(evalplus_ids)
    return clean, evalplus_ids


def write_manifest(splits, clean=None, evalplus_ids=None, output_path="/kaggle/working/mbpp_split_manifest.json"):
    obj = {name: {"count": len(rows), "task_ids": [int(x["task_id"]) for x in rows]}
           for name, rows in splits.items()}
    if clean is not None:
        obj["mbppplus_decontaminated"] = {
            name: {"count": len(rows), "task_ids": [int(r["task_id"]) for r in rows]}
            for name, rows in clean.items()
        }
    if evalplus_ids is not None:
        obj["evalplus_mbppplus_ids"] = sorted(evalplus_ids)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    Path(output_path).write_text(json.dumps(obj, indent=2))
    return output_path


if __name__ == "__main__":
    s = load_mbpp()
    for name, rows in s.items():
        print(f"{name}: {len(rows)} rows; first ID {rows[0]['task_id']}")
    clean, evalplus_ids = exclude_evalplus_overlap(s)
    print("MBPP+ benchmark task IDs:", len(evalplus_ids))
    for name, rows in clean.items():
        print(f"Eligible {name}: {len(rows)} tasks after overlap exclusion")
    print(write_manifest(s, clean, evalplus_ids))


In [ ]:
%%writefile test_runner.py
"""Small, resource-limited function-code runner for Kaggle Day 1.

SECURITY: subprocess + resource limits are NOT a strong sandbox. Model-generated
code can still access local files or network unless the external runtime forbids
it. Never run with secrets, private data or elevated access. For a public-facing
benchmark or adversarial model, run in a truly isolated container/service.
"""
from __future__ import annotations

import json
import os
import resource
import subprocess
import sys
import tempfile
from pathlib import Path

# Entire driver runs in a separate Python interpreter; no generated code is exec'd
# in the notebook kernel. Child stdout is reserved for a JSON response.
_CHILD = r'''
import ast, contextlib, io, json, sys, traceback
payload = json.load(sys.stdin)
source = payload["code"]
checks = payload["tests"]
setup = payload.get("setup", "") or ""

def trace(e):
    return "".join(traceback.format_exception_only(type(e), e)).strip()[:1800]

namespace = {"__name__": "__main__"}
result = {"status": "passed", "passed": 0, "total": len(checks), "feedback": "All visible tests passed", "failures": []}

try:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        exec(compile(source, "<candidate>", "exec"), namespace)
        if setup:
            exec(compile(setup, "<setup>", "exec"), namespace)
except BaseException as e:
    result["status"] = "syntax_error" if isinstance(e, SyntaxError) else "runtime_error"
    result["feedback"] = f"Failed before tests: {trace(e)}"
else:
    for test in checks:
        try:
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                # For simple equality assertions, extract actual/expected values
                # to provide useful execution feedback without reevaluating them.
                tree = ast.parse(test, mode="exec")
                if (len(tree.body) == 1 and isinstance(tree.body[0], ast.Assert) and
                    isinstance(tree.body[0].test, ast.Compare) and
                    len(tree.body[0].test.ops) == 1 and
                    isinstance(tree.body[0].test.ops[0], ast.Eq)):
                    cmp = tree.body[0].test
                    actual = eval(compile(ast.Expression(cmp.left), "<actual>", "eval"), namespace)
                    expected = eval(compile(ast.Expression(cmp.comparators[0]), "<expected>", "eval"), namespace)
                    if actual != expected:
                        raise AssertionError(f"expected {repr(expected)[:220]}, got {repr(actual)[:220]}")
                else:
                    exec(compile(tree, "<test>", "exec"), namespace)
            result["passed"] += 1
        except BaseException as e:
            kind = "wrong_output" if isinstance(e, AssertionError) else "runtime_error"
            result["status"] = kind if result["status"] == "passed" else result["status"]
            result["failures"].append({"test": test[:500], "kind": kind, "message": trace(e)})
            # feedback limited to first failing test to avoid lengthy prompts
            if len(result["failures"]) >= 2:
                break
    if result["failures"]:
        result["feedback"] = "\n".join(
            f"Test: {f['test']}\nError: {f['message']}" for f in result["failures"]
        )[:1600]
print(json.dumps(result, ensure_ascii=False))
'''


def _limits():
    # POSIX-only; Kaggle is Linux. Limits accidental infinite loops and output size.
    resource.setrlimit(resource.RLIMIT_CPU, (3, 4))
    resource.setrlimit(resource.RLIMIT_AS, (1024 * 1024**2, 1024 * 1024**2))
    resource.setrlimit(resource.RLIMIT_FSIZE, (1024 * 1024, 1024 * 1024))


def run_visible_tests(code: str, tests: list[str], setup: str = "", timeout_s: int = 6) -> dict:
    """Execute candidate and visible asserts with timeout.

    Avoid passing hidden benchmark tests here: they must not feed back to the model.
    """
    payload = json.dumps({"code": code, "tests": list(tests), "setup": setup})
    # Pass a small environment, with no authentication tokens inherited.
    env = {"PATH": os.environ.get("PATH", "/usr/bin:/bin"),
           "PYTHONIOENCODING": "utf-8", "PYTHONDONTWRITEBYTECODE": "1"}
    with tempfile.TemporaryDirectory(prefix="gemma_code_") as temp:
        try:
            cp = subprocess.run(
                [sys.executable, "-I", "-S", "-B", "-c", _CHILD],
                input=payload, text=True, stdout=subprocess.PIPE,
                stderr=subprocess.PIPE, cwd=temp,
                env=env, preexec_fn=_limits, timeout=timeout_s,
            )
        except subprocess.TimeoutExpired:
            return {"status": "timeout", "passed": 0, "total": len(tests),
                    "feedback": f"Execution timed out after {timeout_s} seconds", "failures": []}
        if cp.returncode != 0:
            return {"status": "runtime_error", "passed": 0, "total": len(tests),
                    "feedback": f"Runner failed: {cp.stderr[-700:]} (exit={cp.returncode})", "failures": []}
        try:
            output = json.loads(cp.stdout)
            assert output["status"] in {"passed", "wrong_output", "runtime_error", "syntax_error"}
            return output
        except (ValueError, KeyError, AssertionError):
            return {"status": "runner_error", "passed": 0, "total": len(tests),
                    "feedback": f"Malformed runner result: {cp.stdout[-500:]}", "failures": []}


if __name__ == "__main__":
    tests = ["assert find_max([1,5,3]) == 5", "assert find_max([-4, -2]) == -2"]
    wrong = run_visible_tests("def find_max(a):\n    return min(a)\n", tests)
    right = run_visible_tests("def find_max(a):\n    return max(a)\n", tests)
    print("Wrong:", wrong)
    print("Right:", right)
    assert wrong["status"] == "wrong_output"
    assert right["status"] == "passed"


In [ ]:
%%writefile agent_loop.py
"""Gemma 4 multi-turn function-repair loop (day-one research prototype)."""
from __future__ import annotations

import re
from typing import Any

from test_runner import run_visible_tests


def extract_python(text: str) -> str:
    """Best effort; keep a record of raw responses for audit."""
    fences = re.findall(r"```(?:python|py)?\s*\n(.*?)```", text, re.S | re.I)
    if fences:
        return fences[-1].strip()
    text = text.strip().replace("<|im_end|>", "").strip()
    # If Gemma adds a preface, discard it before the first code-like line.
    match = re.search(r"(?m)^(?:from\s+\S+\s+import|import\s+\S+|def\s+\w+|class\s+\w+|@\w+)", text)
    return text[match.start():].strip() if match else text


def build_initial_messages(problem: dict) -> list[dict[str, str]]:
    examples = "\n".join(problem.get("test_list", [])[:3])
    return [
        {"role": "system", "content": "Write correct Python code only. Do not include Markdown or explanations. "
            "Include all necessary imports and define the function exactly as requested."},
        {"role": "user", "content": f"Problem:\n{problem['text']}\n\nVisible tests:\n{examples}\n\nWrite Python code that passes these tests."},
    ]


def run_agent(problem: dict, generate_fn, max_attempts: int = 3, max_new_tokens: int = 384) -> dict:
    """generate_fn(messages, max_new_tokens) -> (response_text, input_tokens, output_tokens)."""
    assert 1 <= max_attempts <= 3
    messages = build_initial_messages(problem)
    history = []
    for attempt in range(1, max_attempts + 1):
        raw, prompt_tokens, completion_tokens = generate_fn(messages, max_new_tokens)
        code = extract_python(raw)
        outcome = run_visible_tests(code, problem["test_list"], problem.get("test_setup_code") or "")
        item = {
            "attempt": attempt, "code": code, "raw_output": raw,
            "status": outcome["status"], "feedback": outcome["feedback"],
            "visible_passed": outcome["passed"], "visible_total": outcome["total"],
            "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens,
        }
        history.append(item)
        if outcome["status"] == "passed":
            break
        messages.append({"role": "assistant", "content": code})
        messages.append({"role": "user", "content": "The previous code FAILED the visible tests.\n"
                        + outcome["feedback"] + "\nProvide a corrected full Python solution. "
                        "Do not repeat the same code. Output code only."})
    return {
        "task_id": int(problem["task_id"]), "problem": problem["text"],
        "solved_visible": history[-1]["status"] == "passed", "history": history,
        "attempts_used": len(history),
        "total_completion_tokens": sum(h["completion_tokens"] for h in history),
        "total_prompt_tokens": sum(h["prompt_tokens"] for h in history),
        "note": "Visible-test success only; not MBPP+ or hidden-test success.",
    }


## C. Validate execution harness before loading the model
The script must report a failure for intentionally incorrect code and a pass for correct code.

In [ ]:
!python test_runner.py
from test_runner import run_visible_tests
assert run_visible_tests("def f(x):\n return x + 1", ["assert f(1)==2"])["status"] == "passed"
assert run_visible_tests("def f(x):\n return x + 1", ["assert f(1)==3"])["status"] == "wrong_output"
assert run_visible_tests("def f(x)\n return x + 1", ["assert f(1)==2"])["status"] == "syntax_error"
print("Harness smoke tests passed")

## D. Download MBPP and exclude all MBPP+ benchmark IDs from training
The official MBPP training split and the newer MBPP+ benchmark **overlap**. We must exclude benchmark IDs to avoid contaminating our later evaluation. MBPP+ is loaded for IDs only; no benchmark reference code or hidden tests are used in training or prompts.

In [ ]:
from prepare_datasets import load_mbpp, exclude_evalplus_overlap, write_manifest
splits = load_mbpp()
clean, mbppplus_ids = exclude_evalplus_overlap(splits)
for split_name in ("train", "validation", "test"):
    print(f"Official MBPP {split_name}: {len(splits[split_name])}")
print(f"MBPP+ benchmark IDs: {len(mbppplus_ids)}")
for split_name in ("train", "validation"):
    print(f"Eligible {split_name} tasks after exclusion: {len(clean[split_name])}")
print("Saved:", write_manifest(splits, clean, mbppplus_ids))
print("Example train problem:", clean["train"][0]["text"])
print("Visible tests:", clean["train"][0]["test_list"][:2])

## E. Load Gemma 4 in 4-bit on GPU 0
Start with E4B. If loading fails from an out-of-memory error or architecture/quantization incompatibility, set `MODEL_ID = "google/gemma-4-E2B-it"` and rerun from this cell after restarting the Kaggle session.

Check the [official Gemma 4 E4B model card](https://huggingface.co/google/gemma-4-E4B-it) if the model requires an updated dependency or authentication.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig
import transformers

MODEL_ID = "google/gemma-4-E4B-it"  # fallback: google/gemma-4-E2B-it
SEED = 42
torch.manual_seed(SEED)

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16, # Tesla T4: prefer float16, not bfloat16
)
print("Transformers:", transformers.__version__)
print("Loading:", MODEL_ID)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    device_map={"": 0},  # one T4; avoid silent multi-GPU sharding
    dtype=torch.float16,
)
model.eval()
print("Model loaded:", MODEL_ID)
print(f"GPU 0 allocated: {torch.cuda.memory_allocated(0)/1024**3:.2f} GiB")

## F. Create the model-generation function
Use Gemma 4 chat formatting with thinking disabled for a fast, comparable coding baseline. Save exact prompt and completion token counts.

In [ ]:
from agent_loop import run_agent

def gemma_generate(messages, max_new_tokens=384):
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    ).to(model.device)
    input_len = int(inputs["input_ids"].shape[1])
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=processor.tokenizer.pad_token_id,
        )
    full = outputs[0]
    generated = full[input_len:]
    text = processor.decode(generated, skip_special_tokens=True)
    return text, input_len, int(generated.shape[0])

print("Generation function ready")

## G. One-problem smoke test
Run a single training example first. If the model formats the code poorly, inspect `raw_output` and `code` before launching a longer baseline.

In [ ]:
one = run_agent(clean["train"][0], gemma_generate, max_attempts=3, max_new_tokens=384)
print("Task:", one["task_id"], "Solved visible tests:", one["solved_visible"])
for step in one["history"]:
    print(f"Attempt {step['attempt']}: {step['status']}; {step['visible_passed']}/{step['visible_total']} tests; {step['completion_tokens']} output tokens")
    print("Code:\n", step["code"][:1000])
    print("Feedback:\n", step["feedback"][:700])

## H. Small baseline on training and validation
Start with 5 + 5. If this finishes and code generation works, increase to 20 + 20 or more. Results are append-saved after each task and skipped on rerun, so you do not lose completed work. **Do not include MBPP test/MBPP+ tasks in this smoke benchmark.**

In [ ]:
import json, random
from pathlib import Path
from datetime import datetime, timezone

N_TRAIN = 5
N_VALIDATION = 5
MAX_ATTEMPTS = 3
MAX_NEW_TOKENS = 384
RESULTS_JSONL = Path("/kaggle/working/baseline_trials_" + MODEL_ID.split("/")[-1] + ".jsonl")

train_sample = random.Random(SEED).sample(clean["train"], N_TRAIN)
valid_sample = random.Random(SEED + 1).sample(clean["validation"], N_VALIDATION)
selected = [("train", p) for p in train_sample] + [("validation", p) for p in valid_sample]

existing = []
if RESULTS_JSONL.exists():
    for line in RESULTS_JSONL.read_text().splitlines():
        if line.strip(): existing.append(json.loads(line))
completed = {(r["model"], r["split"], int(r["task_id"])) for r in existing}

for split_name, problem in selected:
    task_id = int(problem["task_id"])
    if (MODEL_ID, split_name, task_id) in completed:
        print("Already done:", split_name, task_id)
        continue
    row = run_agent(problem, gemma_generate, max_attempts=MAX_ATTEMPTS, max_new_tokens=MAX_NEW_TOKENS)
    row.update({"split": split_name, "model": MODEL_ID,
                "created_utc": datetime.now(timezone.utc).isoformat(),
                "max_attempts": MAX_ATTEMPTS, "max_new_tokens": MAX_NEW_TOKENS})
    with RESULTS_JSONL.open("a") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        f.flush()
    print(f"{split_name} {task_id}: {'PASS' if row['solved_visible'] else 'FAIL'} in {row['attempts_used']} attempts; "
          f"completion tokens={row['total_completion_tokens']}")
print("Checkpointed to:", RESULTS_JSONL)

## I. Final Day 1 baseline summary and JSON export
These are *cumulative pass-within-k rates* over sequential retries with feedback, not traditional independently sampled `pass@k` and not hidden-test accuracy.

In [ ]:
from collections import Counter

all_records = [json.loads(x) for x in RESULTS_JSONL.read_text().splitlines() if x.strip()]
selected_ids = {(split_name, int(p['task_id'])) for split_name,p in selected}
results = [r for r in all_records if r['model']==MODEL_ID and (r['split'],int(r['task_id'])) in selected_ids]
results.sort(key=lambda r: (r['split'], r['task_id']))

def passed_within(row, k):
    return any(a['status']=='passed' for a in row['history'][:k])

summary = {}
for name in ('train', 'validation'):
    subset = [r for r in results if r['split']==name]
    if not subset: continue
    summary[name] = {"num_problems":len(subset),
                     **{f"visible_success_within_{k}":sum(passed_within(r,k) for r in subset)/len(subset)
                        for k in (1,2,3)},
                     "first_attempt_failures":dict(Counter(r['history'][0]['status'] for r in subset)),
                     "mean_completion_tokens":sum(r['total_completion_tokens'] for r in subset)/len(subset)}

output={"model":MODEL_ID, "seed":SEED, "split_manifest":"mbpp_split_manifest.json",
        "benchmark":"Original MBPP visible tests only; MBPP+ IDs excluded from train/validation",
        "hardware":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        "summary":summary, "results":results}
output_path=Path('/kaggle/working/baseline_results.json')
output_path.write_text(json.dumps(output, indent=2))
print(json.dumps(summary, indent=2))
print("Saved", output_path)
print("NOTE: Passing visible tests does not prove hidden-test correctness.")

## Day 1 completion checklist

- [ ] GPU available and model loaded in 4-bit
- [ ] MBPP official splits loaded and MBPP+ overlap removed from train/validation
- [ ] Runner passes intentionally correct/incorrect test cases
- [ ] One training-task smoke test completed with feedback and retries
- [ ] At least 5 training + 5 validation tasks completed
- [ ] `baseline_results.json`, `baseline_trials_*.jsonl`, `mbpp_split_manifest.json` saved
- [ ] Save notebook **with outputs** and back up result files

**Next: Day 2** will build validated failure→correction examples exclusively from the cleaned training split. Final held-out evaluation is a separate later stage.